# Cross-Stress Subject-Level Analysis

**Purpose**: Fix pseudo-replication in statistical testing by aggregating EEG band powers at the **subject level (Patient_ID)** before running Kruskal-Wallis and Jonckheere-Terpstra tests.

In the original `Cross_Stress_Deep_Analysis.ipynb`, each *recording* was treated as an independent observation. Since some patients contribute multiple recordings, this inflates sample size and violates the independence assumption. Here we average all recordings per patient first, yielding one observation per subject per group.

## 1. Setup & Data Loading

In [1]:
import os, pickle
import numpy as np
import pandas as pd
from scipy.stats import (
    ttest_ind, kruskal, norm as scipy_norm,
)
from statsmodels.stats.multitest import multipletests

CH_NAMES = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FT7', 'FC3', 'FCz',
    'FC4', 'FT8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'TP7', 'CP3', 'CPz',
    'CP4', 'TP8', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'Oz', 'O2'
]

BANDS = {'Theta': (4.0, 8.0), 'Alpha': (8.0, 13.0), 'Beta': (13.0, 30.0)}

In [2]:
# Load labels
df = pd.read_csv('../data/comprehensive_labels.csv')
print(f'Total recordings: {len(df)}')
print(f'Unique patients:  {df["Patient_ID"].nunique()}')

# Load cached band powers
CACHE_PATH = '../var/deep_stress_band_powers.pkl'
with open(CACHE_PATH, 'rb') as f:
    all_powers = pickle.load(f)
print(f'Cached band powers: {len(all_powers["Theta"])} recordings')

Total recordings: 70
Unique patients:  17
Cached band powers: 70 recordings


## 2. Cross-Group Assignment

In [3]:
def assign_cross_group(df, dss_threshold=60.0):
    """Assign DASS x DSS cross-groups at a given DSS threshold."""
    df = df.copy()
    df['DASS_class'] = (df['Group'] == 'increase').astype(int)
    df['DSS_class'] = (df['Stress_Score'] >= dss_threshold).astype(int)

    def _assign(row):
        if row['DASS_class'] == 1 and row['DSS_class'] == 1:
            return 'Both-Stress'
        elif row['DASS_class'] == 0 and row['DSS_class'] == 1:
            return 'Acute-Only'
        elif row['DASS_class'] == 0 and row['DSS_class'] == 0:
            return 'Both-Normal'
        else:
            return 'Chronic-Only'

    df['Cross_Group'] = df.apply(_assign, axis=1)
    return df

df = assign_cross_group(df, dss_threshold=60.0)
print('Cross-group counts (DSS threshold=60):')
print(df['Cross_Group'].value_counts())
print()

# Show subject counts per group
for g in ['Both-Stress', 'Acute-Only', 'Both-Normal', 'Chronic-Only']:
    sub = df[df['Cross_Group'] == g]
    print(f'  {g:15s}: {len(sub):3d} recordings, {sub["Patient_ID"].nunique():3d} unique patients')

Cross-group counts (DSS threshold=60):
Cross_Group
Both-Normal     29
Acute-Only      27
Both-Stress     13
Chronic-Only     1
Name: count, dtype: int64

  Both-Stress    :  13 recordings,   5 unique patients
  Acute-Only     :  27 recordings,  10 unique patients
  Both-Normal    :  29 recordings,  11 unique patients
  Chronic-Only   :   1 recordings,   1 unique patients


## 3. Subject-Level Aggregation (Core Fix)

Instead of treating each recording as independent, we average all recordings per `Patient_ID` within each group. This yields one observation per subject, eliminating pseudo-replication.

In [4]:
def get_subject_group_data(df_sub, all_powers, band, group_col='Cross_Group'):
    """Extract band power arrays aggregated at the subject (Patient_ID) level.

    For each group, recordings belonging to the same patient are averaged
    into a single (30,) vector, so the returned array has shape
    (N_unique_patients_in_group, 30) instead of (N_recordings, 30).
    """
    result = {}
    for g in df_sub[group_col].unique():
        group_df = df_sub[df_sub[group_col] == g]
        patient_vectors = []

        for pid, pat_df in group_df.groupby('Patient_ID'):
            rids = pat_df['Recording_ID'].astype(str).tolist()
            rec_arrays = [all_powers[band][r] for r in rids if r in all_powers[band]]
            if len(rec_arrays) == 0:
                continue
            # Average across recordings for this patient -> (30,)
            patient_mean = np.mean(rec_arrays, axis=0)
            patient_vectors.append(patient_mean)

        if patient_vectors:
            result[g] = np.stack(patient_vectors)  # (N_patients, 30)
        else:
            result[g] = np.array([]).reshape(0, 30)

    return result

print('get_subject_group_data() defined.')

get_subject_group_data() defined.


## 4. Statistical Testing Helpers

In [5]:
def cohens_d(a, b):
    """Compute Cohen's d effect size."""
    return (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2) / 2)


def jonckheere_terpstra(ordered_groups):
    """Manual Jonckheere-Terpstra trend test for ordered groups."""
    n_concordant = 0
    for i in range(len(ordered_groups)):
        for j in range(i + 1, len(ordered_groups)):
            for xi in ordered_groups[i]:
                for xj in ordered_groups[j]:
                    if xj > xi:
                        n_concordant += 1
                    elif xj == xi:
                        n_concordant += 0.5
    JT = n_concordant
    n_list = [len(g) for g in ordered_groups]
    N = sum(n_list)
    E_JT = (N**2 - sum(n**2 for n in n_list)) / 4
    V_JT = (N**2 * (2*N + 3) - sum(n**2 * (2*n + 3) for n in n_list)) / 72
    Z_JT = (JT - E_JT) / np.sqrt(V_JT)
    p_JT = 2 * (1 - scipy_norm.cdf(abs(Z_JT)))
    return Z_JT, p_JT


def run_2group_stats(g1, g2, name1, name2, band_name, ch_names=CH_NAMES):
    """Run 2-group comparison: global + per-channel."""
    m1, m2 = g1.mean(axis=1), g2.mean(axis=1)
    H, p_kw = kruskal(m1, m2)
    t, p_t = ttest_ind(m1, m2, equal_var=False)
    d = cohens_d(m1, m2)

    p_vals = []
    for ch in range(g1.shape[1]):
        _, p = kruskal(g1[:, ch], g2[:, ch])
        p_vals.append(p)
    _, p_fdr, _, _ = multipletests(p_vals, method='fdr_bh')
    n_sig = (p_fdr < 0.05).sum()

    return {
        'condition': f'{name1} vs {name2}',
        'band': band_name,
        'n1': g1.shape[0], 'n2': g2.shape[0],
        'mean1': m1.mean(), 'mean2': m2.mean(),
        'KW_p': p_kw, 't_p': p_t, 'Cohen_d': d,
        'n_sig_ch': n_sig,
        'p_fdr_channels': p_fdr,
    }


def run_3group_stats(g1, g2, g3, names, band_name, ch_names=CH_NAMES):
    """Run 3-group comparison: global KW + pairwise + JT dose-response."""
    m1, m2, m3 = g1.mean(axis=1), g2.mean(axis=1), g3.mean(axis=1)

    H, p_kw = kruskal(m1, m2, m3)

    pairs = [(names[0], names[1], m1, m2),
             (names[0], names[2], m1, m3),
             (names[1], names[2], m2, m3)]
    pairwise = {}
    for na, nb, a, b in pairs:
        t, p = ttest_ind(a, b, equal_var=False)
        d = cohens_d(a, b)
        pairwise[f'{na}_vs_{nb}'] = {'t': t, 'p': p, 'd': d}

    p_vals_ch = []
    for ch in range(g1.shape[1]):
        _, p = kruskal(g1[:, ch], g2[:, ch], g3[:, ch])
        p_vals_ch.append(p)
    _, p_fdr_ch, _, _ = multipletests(p_vals_ch, method='fdr_bh')
    n_sig = (p_fdr_ch < 0.05).sum()

    # Jonckheere-Terpstra: BN < AO < BS
    Z_JT, p_JT = jonckheere_terpstra([m3, m2, m1])
    monotonic = m1.mean() > m2.mean() > m3.mean()

    return {
        'band': band_name,
        'KW_H': H, 'KW_p': p_kw,
        'pairwise': pairwise,
        'JT_Z': Z_JT, 'JT_p': p_JT, 'monotonic': monotonic,
        'n_sig_ch': n_sig,
        'p_fdr_channels': p_fdr_ch,
        'means': {names[0]: m1.mean(), names[1]: m2.mean(), names[2]: m3.mean()},
    }

print('Statistical helpers defined.')

Statistical helpers defined.


## 5. Subject-Level 3-Group Analysis (BS vs AO vs BN)

In [6]:
results_3g = {}

for band in BANDS:
    groups = get_subject_group_data(df, all_powers, band)
    g_BS = groups.get('Both-Stress', np.array([]).reshape(0, 30))
    g_AO = groups.get('Acute-Only', np.array([]).reshape(0, 30))
    g_BN = groups.get('Both-Normal', np.array([]).reshape(0, 30))

    print(f'\n{"="*70}')
    print(f'  {band} Band  —  Subject-Level Analysis')
    print(f'{"="*70}')
    print(f'  Both-Stress (BS): {g_BS.shape[0]} subjects')
    print(f'  Acute-Only  (AO): {g_AO.shape[0]} subjects')
    print(f'  Both-Normal (BN): {g_BN.shape[0]} subjects')

    res = run_3group_stats(g_BS, g_AO, g_BN,
                           ['Both-Stress', 'Acute-Only', 'Both-Normal'], band)
    results_3g[band] = res

    # --- Print summary ---
    print(f'\n  Global Kruskal-Wallis:  H = {res["KW_H"]:.3f},  p = {res["KW_p"]:.4f}')
    sig_kw = '***' if res['KW_p'] < 0.001 else '**' if res['KW_p'] < 0.01 else '*' if res['KW_p'] < 0.05 else 'ns'
    print(f'  Significance: {sig_kw}')

    print(f'\n  Group means (global avg across 30 ch):')
    for name, val in res['means'].items():
        print(f'    {name:15s}: {val:.4f}')

    print(f'\n  Pairwise Cohen\'s d:')
    for pair_name, pair_stats in res['pairwise'].items():
        sig = '*' if pair_stats['p'] < 0.05 else 'ns'
        print(f'    {pair_name:35s}:  d = {pair_stats["d"]:+.3f},  p = {pair_stats["p"]:.4f}  ({sig})')

    print(f'\n  Jonckheere-Terpstra (BN < AO < BS):')
    print(f'    Z = {res["JT_Z"]:.3f},  p = {res["JT_p"]:.4f}')
    print(f'    Monotonic trend (BS > AO > BN): {res["monotonic"]}')

    print(f'\n  Per-channel FDR (q < 0.05): {res["n_sig_ch"]} / 30 channels significant')


  Theta Band  —  Subject-Level Analysis
  Both-Stress (BS): 5 subjects
  Acute-Only  (AO): 10 subjects
  Both-Normal (BN): 11 subjects

  Global Kruskal-Wallis:  H = 3.051,  p = 0.2175
  Significance: ns

  Group means (global avg across 30 ch):
    Both-Stress    : 3.6318
    Acute-Only     : 3.5908
    Both-Normal    : 3.0987

  Pairwise Cohen's d:
    Both-Stress_vs_Acute-Only          :  d = +0.054,  p = 0.9287  (ns)
    Both-Stress_vs_Both-Normal         :  d = +0.864,  p = 0.2224  (ns)
    Acute-Only_vs_Both-Normal          :  d = +0.731,  p = 0.1366  (ns)

  Jonckheere-Terpstra (BN < AO < BS):
    Z = 1.649,  p = 0.0992
    Monotonic trend (BS > AO > BN): True

  Per-channel FDR (q < 0.05): 0 / 30 channels significant

  Alpha Band  —  Subject-Level Analysis
  Both-Stress (BS): 5 subjects
  Acute-Only  (AO): 10 subjects
  Both-Normal (BN): 11 subjects

  Global Kruskal-Wallis:  H = 1.408,  p = 0.4945
  Significance: ns

  Group means (global avg across 30 ch):
    Both-Stress  

## 6. Summary Table

In [7]:
rows = []
for band, res in results_3g.items():
    bs_vs_bn = res['pairwise']['Both-Stress_vs_Both-Normal']
    bs_vs_ao = res['pairwise']['Both-Stress_vs_Acute-Only']
    ao_vs_bn = res['pairwise']['Acute-Only_vs_Both-Normal']
    rows.append({
        'Band': band,
        'KW p': f'{res["KW_p"]:.4f}',
        'd(BS-BN)': f'{bs_vs_bn["d"]:+.3f}',
        'p(BS-BN)': f'{bs_vs_bn["p"]:.4f}',
        'd(BS-AO)': f'{bs_vs_ao["d"]:+.3f}',
        'p(BS-AO)': f'{bs_vs_ao["p"]:.4f}',
        'd(AO-BN)': f'{ao_vs_bn["d"]:+.3f}',
        'p(AO-BN)': f'{ao_vs_bn["p"]:.4f}',
        'JT Z': f'{res["JT_Z"]:.3f}',
        'JT p': f'{res["JT_p"]:.4f}',
        'Monotonic': res['monotonic'],
        'FDR ch': res['n_sig_ch'],
    })

df_summary = pd.DataFrame(rows)
print('\n=== Subject-Level 3-Group Summary (DSS threshold=60) ===\n')
print(df_summary.to_string(index=False))


=== Subject-Level 3-Group Summary (DSS threshold=60) ===

 Band   KW p d(BS-BN) p(BS-BN) d(BS-AO) p(BS-AO) d(AO-BN) p(AO-BN)  JT Z   JT p  Monotonic  FDR ch
Theta 0.2175   +0.864   0.2224   +0.054   0.9287   +0.731   0.1366 1.649 0.0992       True       0
Alpha 0.4945   +0.786   0.1867   +0.484   0.4006   +0.216   0.6447 1.171 0.2417       True       0
 Beta 0.6781   +0.561   0.3340   +0.265   0.6310   +0.181   0.6998 0.788 0.4304       True       0
